# 20 — Gold Target Engineering

**Objetivo:** congelar y materializar el target V1 de ARGOS con semántica explícita, horizonte temporal exacto y censura correcta.

Este notebook consume exclusivamente la capa Silver cerrada en M2. No crea features ni splits.

## Semantic freeze V1

`y_addition_180d = 1` significa que existe al menos un evento canónico `ADICION EN EL VALOR` en la ventana **(t0, t0 + 180 días]**, donde `t0 = fecha_de_firma`.

Los contratos sin seguimiento completo de 180 días son **censurados** y no se convierten en negativos.

In [6]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m2_gate = paths.docs / "decisions" / "M2_GATE.md"
assert m2_gate.exists() and "M2 ✅ CLOSED" in m2_gate.read_text(encoding="utf-8"), (
    "M2 no está formalmente cerrado. No iniciar Gold."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")

{
    "gold_schema_version": GOLD_SPEC.schema_version,
    "target": GOLD_SPEC.target.name,
    "horizon_days": GOLD_SPEC.target.horizon_days,
    "event_scope": GOLD_SPEC.target.included_event_types,
    "duckdb_threads": GOLD_SPEC.runtime.duckdb_threads,
    "duckdb_memory_limit": GOLD_SPEC.runtime.duckdb_memory_limit,
    "m2_gate": str(m2_gate),
}

from argos.gold.target import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION

'm3-20-target-engineering-v1'

In [7]:
from argos.gold.target import build_target_gold

result = build_target_gold(
    silver_dir=paths.silver,
    gold_dir=paths.gold,
    reports_root=paths.reports,
    manifest_dir=paths.root / "artifacts" / "manifests",
    gold_spec=GOLD_SPEC,
    temp_directory=paths.root / ".cache" / "duckdb",
    overwrite=True,
)

{name: str(path.relative_to(paths.root)) for name, path in result.artifacts.items()}

[20] materializing target population and censoring state...
[20] materializing labels from future qualifying events only...


{'target_population': 'data/gold/target_population.parquet',
 'labels': 'data/gold/labels/y_addition_180d.parquet'}

## Auditoría del target

In [8]:
display(result.event_scope)
display(result.censoring)
display(result.by_year)

,event_type,canonical_events,distinct_contracts,min_event_at,max_event_at,included_in_target
0,MODIFICACION GENERAL,3616766,2024452,2015-01-01 00:00:00-05:00,2026-09-09 00:00:00-05:00,False
1,No definido,2470594,2090124,2015-09-02 00:00:00-05:00,2026-09-09 00:00:00-05:00,False
2,CONCLUSION,1968834,1914612,2016-01-01 00:00:00-05:00,2026-09-09 00:00:00-05:00,False
3,ADICION EN EL VALOR,240204,198072,2015-01-01 00:00:00-05:00,2026-09-09 00:00:00-05:00,True
4,REACTIVACIoN,127463,89849,2016-01-01 00:00:00-05:00,2026-09-09 00:00:00-05:00,False
5,SUSPENSIoN,116340,81192,2016-01-01 00:00:00-05:00,2026-09-09 00:00:00-05:00,False
6,CESION,95980,87243,2016-01-01 00:00:00-05:00,2026-09-09 00:00:00-05:00,False
7,EXTENSION,352,341,2016-01-01 00:00:00-05:00,2018-01-03 00:00:00-05:00,False
8,REDUCCION EN EL VALOR,132,130,2016-01-01 00:00:00-05:00,2017-09-08 00:00:00-05:00,False
9,EXPIRACION,1,1,2017-02-01 00:00:00-05:00,2017-02-01 00:00:00-05:00,False


,population_status,contracts
0,eligible,5312182
1,missing_t0,426452
2,right_censored,301370
3,t0_after_label_cutoff,26726


,t0_year,eligible,positives,negatives,prevalence
0,2015,10,1,9,0.100000
1,2016,1342,19,1323,0.014158
2,2017,22259,859,21400,0.038591
3,2018,142973,9474,133499,0.066264
4,2019,142592,8036,134556,0.056357
5,2020,357251,9682,347569,0.027101
6,2021,561581,5803,555778,0.010333
7,2022,710534,9289,701245,0.013073
8,2023,843059,15283,827776,0.018128
9,2024,950670,9180,941490,0.009656


In [9]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,target_grain_unique,grain,True,True,"{""distinct_contracts"": 6066730, ""rows"": 6066730}",one Gold target row per contract_id
1,t0_coverage,temporal,True,True,"{""coverage"": 0.9297064481194977, ""rows"": 60667...",t0 coverage >= 85.00%
2,t0_plausible,temporal,True,True,"{""implausible_t0"": 0, ""plausible_year_max"": 21...",all parsed t0 values used by Gold fall inside ...
3,event_scope_explicit_and_nonempty,target_semantics,True,True,"{""canonical_events"": 240204, ""included_event_t...",configured target event types exist in canonic...
4,future_events_only,point_in_time,True,True,"{""violations"": 0}",every qualifying target event occurs strictly ...
5,horizon_exact,point_in_time,True,True,"{""horizon_days"": 180, ""violations"": 0}","qualifying events are within (t0, t0+180d]"
6,censoring_correct,censoring,True,True,"{""censored_rows"": 301370, ""labeled_censored_ro...",right-censored contracts never receive a negat...
7,labels_binary_or_null,target,True,True,"{""eligible"": 5312182, ""invalid_eligible_labels...",every eligible contract has a binary label and...
8,label_event_consistency,target,True,True,"{""negative_with_event"": 0, ""positive_without_e...",positive/negative labels agree with qualifying...
9,no_post_cutoff_positive_events,temporal,True,True,"{""violations"": 0}",no positive label is supported by an event aft...


In [10]:
blocking = result.validation.loc[result.validation["blocking"]]
failed = blocking.loc[~blocking["passed"], "check_id"].tolist()
print("20 ✅ PASS" if result.ready_for_next_notebook else "20 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
if failed:
    print("Failed:", ", ".join(failed))
assert result.ready_for_next_notebook, "Target Gold no supera sus checks bloqueantes."

20 ✅ PASS
Blocking checks: 13/13
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/gold_target_y_addition_180d.json


## Criterio de salida

No avanzar a `21_temporal_split_design.ipynb` si `20` no termina en `✅ PASS`. El target queda congelado documentalmente en `docs/target-specs/y_addition_180d.md`.